# PathVQA Filtering Funnel (Figure D3)

Replaces the draft's Figure 6, which plotted three grouped bars and **carried a wrong
number**: it showed 1,504 filtered images where the pickles give 1,241. Its QA counts
(4,630 / 4,362) were right.

The underlying problem is that Figure 6 showed a *two*-stage comparison (raw vs
filtered) of a *three*-stage funnel, so the word-count floor - which removes 86% of the
surviving open-ended pairs - was invisible. This version counts every stage from the
filtering code's own outputs.


In [ ]:
import os
import sys
import pickle
import collections

sys.path.insert(0, os.path.abspath("."))

import matplotlib.pyplot as plt
import numpy as np

import nature_style as ns

ns.apply_style()

FILTERED = os.path.join("..", "pathvqa_data_filtered", "qas")

pairs = []
for split in ("train", "val", "test"):
    with open(os.path.join(FILTERED, split, f"{split}_qa.pkl"), "rb") as handle:
        pairs.extend(pickle.load(handle))

images = {row["image"] for row in pairs}
YES_NO = {"yes", "no"}


def words(text):
    return len(str(text).split())


close_ended = [r for r in pairs if str(r["answer"]).strip().lower() in YES_NO]
open_ended = [r for r in pairs if str(r["answer"]).strip().lower() not in YES_NO]

# The word-count floor, exactly as data_processing applies it: open-ended needs >= 5
# words in BOTH question and answer; close-ended needs >= 5 in the question only,
# because the answer is always "yes" or "no".
oe_kept = [r for r in open_ended if words(r["question"]) >= 5 and words(r["answer"]) >= 5]
ce_kept = [r for r in close_ended if words(r["question"]) >= 5]

print(f"H&E-filtered: {len(images)} images, {len(pairs)} pairs")
print(f"  close-ended {len(close_ended)}  open-ended {len(open_ended)}")
print(f"  after word floor: CE {len(ce_kept)}  OE {len(oe_kept)}")


In [ ]:
# Published starting point (He et al. 2020), not recomputable from the filtered pickles.
RAW_IMAGES, RAW_PAIRS = 4998, 32795   # released files; the publication says 32,799

fig = plt.figure(figsize=ns.mm(ns.SINGLE_COL_MM * 1.55, 78))
ax = fig.add_subplot(111)

# A funnel, not grouped bars: each stage is a share of the one above it, so the reader
# sees what each filter removed rather than only the endpoints.
STAGES = [
    ("Raw PathVQA", RAW_PAIRS, "#BFBFBF"),
    ("H&E-stained\n(>10 nuclei)", len(pairs), "#9A9A9A"),
    ("Word-count floor\n(>=5 words)", len(oe_kept) + len(ce_kept),
     ns.DATASET_COLORS["PathOPEN"]),
]
y = np.arange(len(STAGES))
values = [value for _, value, _ in STAGES]
ax.barh(y, values, 0.52, color=[colour for _, _, colour in STAGES])
ax.set_yticks(y)
ax.set_yticklabels([label for label, _, _ in STAGES], fontsize=ns.fs(7.5))
ax.invert_yaxis()
ax.set_xlabel("Question-answer pairs")
ax.set_xlim(0, RAW_PAIRS * 1.30)

for index, (label, value, _) in enumerate(STAGES):
    share = 100.0 * value / RAW_PAIRS
    ax.text(value + RAW_PAIRS * 0.015, index,
            f"{value:,}  ({share:.0f}% of raw)", va="center", fontsize=ns.fs(7),
            color="#333333")

# Composition of each surviving stage, stated rather than drawn: a second axis would
# double the panel for six numbers.
# Composition stated beside each bar, inside the axes, so nothing runs past the frame.
ax.text(RAW_PAIRS * 0.30, 1.34,
        f"{len(images):,} images  |  {len(close_ended):,} close-ended  "
        f"+ {len(open_ended):,} open-ended",
        fontsize=ns.fs(6.8), color="#666666", va="top")
ax.text(RAW_PAIRS * 0.30, 2.34,
        f"{len(ce_kept):,} close-ended  +  {len(oe_kept):,} open-ended   "
        f"({100.0 * (1 - len(oe_kept) / len(open_ended)):.0f}% of open-ended pairs "
        f"fall below the floor)",
        fontsize=ns.fs(6.8), color="#666666", va="top")
ax.set_ylim(2.62, -0.55)

for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.tick_params(axis="y", length=0)

paths = ns.save(fig, "figD3_pathvqa_funnel")
print("wrote:", paths)
plt.show()


## Draft caption

**Fig. D3 | PathVQA filtering funnel.**
Two filters reduce PathVQA to the subset comparable with PathOPEN. H&E selection
(HistoCartography nuclei detection, >10 nuclei) retains 1,241 of 4,998 images and 8,992
of 32,799 pairs. A word-count floor of five words - applied to both question and answer
for open-ended pairs, and to the question alone for close-ended, whose answer is always
yes or no - leaves 612 open-ended and 3,005 close-ended pairs, about 11% of the original.
The open-ended stage is the severe one: 86% of H&E-surviving open-ended pairs fall below
the floor, which is the quantitative form of the length difference in Fig. D2d.
